# Conformal Prediction Quickstart

Build 90% split conformal prediction intervals and sets with NumPy and scikit-learn. The calibration and test examples must be exchangeable for the finite-sample marginal coverage guarantee. Empirical coverage on a particular test split varies.

Run the code cells in order. Both examples keep the calibration and test data out of model training.

## Regression with synthetic data

Calibrate absolute residuals using the one-based order statistic at rank `ceil((n_cal + 1) * (1 - alpha))`. If the rank exceeds the sample size, use an infinite threshold. Apply the resulting radius to each test prediction.

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X, y = make_regression(n_samples=1500, n_features=6, noise=20, random_state=7)
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, random_state=7)
X_cal, X_test, y_cal, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=7)

model = LinearRegression().fit(X_train, y_train)
scores = np.abs(y_cal - model.predict(X_cal))
alpha = 0.1
rank = int(np.ceil((len(scores) + 1) * (1 - alpha)))
radius = np.partition(scores, rank - 1)[rank - 1] if rank <= len(scores) else np.inf

predictions = model.predict(X_test)
lower, upper = predictions - radius, predictions + radius
print(f"Test coverage: {np.mean((lower <= y_test) & (y_test <= upper)):.1%}")
print(f"Mean interval width: {np.mean(upper - lower):.2f}")

## Classification on handwritten digits

Score the true calibration label with one minus its predicted probability. Include each test label whose score is at or below the calibrated threshold. Report both test coverage and mean prediction set size.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, random_state=7)
X_cal, X_test, y_cal, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=7)

classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
classifier.fit(X_train, y_train)
# The columns of predict_proba are in classes_ order.
cal_columns = np.searchsorted(classifier.classes_, y_cal)
scores = 1 - classifier.predict_proba(X_cal)[np.arange(len(y_cal)), cal_columns]
alpha = 0.1
rank = int(np.ceil((len(scores) + 1) * (1 - alpha)))
threshold = np.partition(scores, rank - 1)[rank - 1] if rank <= len(scores) else np.inf

prediction_sets = (1 - classifier.predict_proba(X_test)) <= threshold
test_columns = np.searchsorted(classifier.classes_, y_test)
covered = prediction_sets[np.arange(len(y_test)), test_columns]
print(f"Test coverage: {np.mean(covered):.1%}")
print(f"Mean set size: {np.mean(prediction_sets.sum(axis=1)):.2f}")

## Next steps

See the [resource catalogue](../README.md) for tutorials, libraries, and methods for other tasks.